![image_1773205728911.png](./image_1773205728911.png "image_1773205728911.png")

![image_1773205929294.png](./image_1773205929294.png "image_1773205929294.png")

Advantages :

![image_1773205968973.png](./image_1773205968973.png "image_1773205968973.png")

**DLT (Delta Live Tables):**  
DLT is a declarative framework where you specify *what* you want to achieve (data transformations), not *how* to achieve it. DLT manages data pipelines, quality, and lineage automatically.

**Job:**  
A Job runs custom Python, SQL, or notebook files and manages dependencies and scheduling. Jobs are used for general workflow automation, including running DLT pipelines or other tasks.

**Key Difference:**  
DLT is focused on data pipeline management and transformation, while Jobs are for orchestrating and scheduling any tasks, including DLT pipelines.

Core Components of DLT
![image_1773206071268.png](./image_1773206071268.png "image_1773206071268.png")

Here we can see the Asset Browser, it has Root folder in it, we can move this from here not from workspace , it'll sync from here

Source Code Files like transformations, we can register it , we can see in transformation folder

We have explorations : in Exploration not directly alligned to code, but it's analysis or alligned with project , it'll not run this file, non source code folder

Utilities : Wheel file , readme etc.

![image_1773206662892.png](./image_1773206662892.png "image_1773206662892.png")

Just create a bascis.py in the transformations to understand core components of DLT

for Streaming table source should be append only , else if it chnages we have to use skipchangesCommits

Also for checking data we can have notebooks in exploration
![image_1773222101729.png](./image_1773222101729.png "image_1773222101729.png")

In [0]:
#CREATE A STREAMING TABLE ---- STREAMING TABLE

import dlt 

@dlt.table (
    name = 'demo_stream_table'  # this will be created in the catalog
)

def demo_stream_table():
    df = spark.readStream.table("databricksharis.silver.sales_enr")
    return df

#CREATE A MATERIALIZED VIEW ----BATCH TABLE

@dlt.table (
    name = 'demo_mat_view'
)
def demo_mat_view():
    df = spark.read.table("databricksharis.silver.sales_enr")
    return df

#CREATE A TEMPORARY BATCH TABLE 

@dlt.view (
    name = 'demo_temp_view'
)
def demo_temp_view():
    df = spark.read.table("databricksharis.silver.sales_enr")
    return df

#CREATE A Temporray Stream TABLE 

@dlt.view (
    name = 'demo_stream_view'
)
def demo_stream_view():
    df = spark.readStream.table("databricksharis.silver.sales_enr")
    return df


### APPEND FLOW
- SOURCE1 ------ CREATE A STREAMING TABLE 1
- SOURCE2 ------ CREATE A STREAMING TABLE 2 

NOW WHAT IF WE HAVE TO PRODUCE 1 DESTINATION TABLE BUT ON TWO DIFFERENT FLOWS, IT'S POSSIBLE USING APPEND FLOW 

- CREATE A EMPTY STREAMINING TABLE
- CREATE A STREAMING QUERIES 
- USE @dlt.appendflow instead of table into some target location (empty straming table)
- biggest advantnages : Ideompotency and incremental ingestion



![image_1773224951235.png](./image_1773224951235.png "image_1773224951235.png")

In [0]:
%sql
Create  two Directores in autovol -----> Flow1 and Flow2
creating all the files in dlt_pipleline

![image_1773244367374.png](./image_1773244367374.png "image_1773244367374.png")
1 now becomes Clothing 

![image_1773244412531.png](./image_1773244412531.png "image_1773244412531.png")

![image_1773244520872.png](./image_1773244520872.png "image_1773244520872.png")

![image_1773244557409.png](./image_1773244557409.png "image_1773244557409.png")

Create a demo table in exploration file

%sql
--create a demo table for Auto cdc

create table databricksharis.bronze.source(
  product_id string,
  product_name string,
  processdate timestamp
)

%sql 
INSERT INTO databricksharis.bronze.source values
(1, "Honey", current_timestamp()),
(2, "milk", current_timestamp()),
(3, "bread", current_timestamp())

In [0]:
import dlt 
from pyspark.sql.functions import *

# STAGING TABLE 
@dlt.table(
    name="scd1_stg"
)
def scd1_stg():
    df = spark.readStream.table("databricksharis.bronze.source")
    return df

# CREATE AN EMPTY STREAMING TABLE 
dlt.create_streaming_table(
    name="scd1_table"
)

dlt.create_auto_cdc_flow(
    target="scd1_table",
    source="scd1_stg",
    keys=["product_id"],
    sequence_by=col("processdate"),
    stored_as_scd_type=1
)

![image_1773245574668.png](./image_1773245574668.png "image_1773245574668.png")

### EXPECTATIONS : DATA QUALITY/ DATA CONSTRAINT
![image_1773247278775.png](./image_1773247278775.png "image_1773247278775.png")

![image_1773247462949.png](./image_1773247462949.png "image_1773247462949.png")

### DLT END TO END 

stream using Autoloader integrated with DLT


LANDING_ZONE: RAW VOLUME

CREATE A FOLDER INSIDE TRANSFORMATIONS 

1. BRONZE 
INSIDE THIS CREATE A FILE INGESTION.PY
2. SILVER
3. SILVER


![image_1773308738084.png](./image_1773308738084.png "image_1773308738084.png")


Here B refers to bronze , we dind't make them as source Table for Silver,as These will be replica of Source only, that's why we have created views in which we did transformations



For Gold : Sources will also be views, because streaming tables can be change, if we want to create a streaming table, source will be append only. As views bring new changes data from bronze, data can be change in the silver tables, Silver Streaming Tables can be used by mostly Data scientits, gold can be used by data analyst, Bi developers etc.

![image_1773311619671.png](./image_1773311619671.png "image_1773311619671.png")

After Creating the Pipeline just upload new updated files in theraw folders, just to check the behaviour of the pipeline

Let's now check the data after running the pipeline


![image_1773331698857.png](./image_1773331698857.png "image_1773331698857.png")

In [0]:
%sql
select * from databricksharis.gold.dim_products where product_id=29

![image_1773331854830.png](./image_1773331854830.png "image_1773331854830.png")

In continous, it'll run only when underlying tables receives some data, it runs the pipeline automatically 

![image_1773332056621.png](./image_1773332056621.png "image_1773332056621.png")

We can create different DLT's as merging into one will create a chaos, like this we can manage the DLT's easily